### Imports

In [ ]:
import torch
import numpy as np
import pandas as pd
from src.preprocessing.sort_tec_data import build_sorted_dataset
from src.preprocessing.dataset import prepare_dataset
from src.configs.config import (
    BASE_DIR_2, OUTPUT_DIR_2,
    F_L1, F_L5, EVENT_THRESHOLD_PERCENTILE, DELAY_EVENT_THRESHOLDS_M,
    PLOTS_DIR_2, GENERATED_PLOTS_DIR, get_date_label,
)
from src.training.lstm_training import train_lstm
from src.training.transformer_training import train_transformer

from src.plots.loss_plots import (
    plot_lstm_loss, plot_transformer_loss,
)
from src.plots.prediction_plots import (
    plot_lstm_prediction, plot_transformer_prediction
)
from src.plots.delay_plots import (
    plot_lstm_delay_l1_l5_combined,
    plot_transformer_delay_l1_l5_combined,
)
from src.utils.iono_delay import delay_metrics, tec_to_iono_delay
from src.utils.save_results import save_delay_csv
from src.utils.metrics import (
    rmse,
    pearson_correlation,
    probability_of_detection,
    critical_success_index,
    f1_score,
    false_alarm_ratio,
)


---
# Dataset 2 — Day 40/41 Forecasting Workflow
---

### Base Directory and Output Setup — Dataset 2

In [ ]:
print(f"Base directory 2  : {BASE_DIR_2}")
print(f"Output directory 2: {OUTPUT_DIR_2}")

### Creating Sorted TEC Dataset — Dataset 2
```text
Process folders iisc0910_TECU to iisc1310_TECU and create one sorted CSV
per folder containing the maximum TEC value for each minute.
Returns: Number of output CSV files written.
```

In [ ]:
written_files_2 = build_sorted_dataset(2)

print("TEC data sorting completed — Dataset 2.")
print(f"Number of output CSV files written: {written_files_2}")
print(f"Saved sorted CSV files to: {OUTPUT_DIR_2}")

#### QUICK SANITY CHECK — Dataset 2
> Print first 6 lines of the first output file

In [ ]:
sample_file_2 = OUTPUT_DIR_2 / "iisc0910_TECU_sorted.csv"
if sample_file_2.exists():
    with sample_file_2.open("r", newline="", encoding="utf-8") as f:
        for i, line in enumerate(f):
            print(line.strip())
            if i >= 5:
                break
else:
    print(f"Sample file not found: {sample_file_2}")

### Data Loading and Preprocessing — Dataset 2
```text
Loads sorted CSVs into a (days × minutes) numpy matrix,
builds supervised input/target pairs, and
normalises using training stats only.
```

In [ ]:
X_train_2, y_train_2, X_val_2, y_val_2, stats_2, daily_matrix_2, daily_files_2 = prepare_dataset(2)

print(f"X_train : {X_train_2.shape}   y_train : {y_train_2.shape}")
print(f"X_val   : {X_val_2.shape}     y_val   : {y_val_2.shape}")
print(f"Norm stats : {stats_2}")

### LSTM Training Workflow — Dataset 2: Day 40 to Day 41 Forecast
> Dataset consists of 41 chronologically sorted TEC files.

```text
Training Set
------------
Input  : Days  1-29
Target : Days  2-30

Validation Set
--------------
Input  : Days 30-40
Target : Days 31-41

Forecast
--------
Input  : Day 39  →  Output : Day 40  (9 May 2024)
Input  : Day 40  →  Output : Day 41  (10 May 2024)
```

In [ ]:
lstm_model_2, lstm_history_2 = train_lstm(X_train_2, y_train_2, X_val_2, y_val_2)

### LSTM Loss and Validation Curve — Dataset 2

In [ ]:
plot_lstm_loss(lstm_history_2, dataset_label="dataset2", output_dir=str(PLOTS_DIR_2))

### LSTM Forecast for Day 40 — 9 May 2024 (Dataset 2)


In [ ]:
x_mean_2, x_std_2 = stats_2["x_mean"], stats_2["x_std"]
y_mean_2, y_std_2 = stats_2["y_mean"], stats_2["y_std"]

day39_raw_2 = daily_matrix_2[38]
actual40_2  = daily_matrix_2[39]

day39_in_2 = ((day39_raw_2 - x_mean_2) / x_std_2)[np.newaxis, ..., np.newaxis]   # (1, 1440, 1)

device = next(lstm_model_2.parameters()).device
src39  = torch.tensor(day39_in_2, dtype=torch.float32).to(device)

lstm_model_2.eval()
with torch.no_grad():
    pred40_norm_2 = lstm_model_2(src39).squeeze().cpu().numpy()   # (1440,)

lstm_pred40_2 = pred40_norm_2 * y_std_2 + y_mean_2

# derive label automatically from the loaded files
label_40_2 = f"Actual Day 40 ({daily_files_2[-2].stem.replace('_sorted', '')})"  # -> "Actual Day 40 (iisc1300_TECU)"

lstm_rmse_40_2, lstm_mae_40_2 = plot_lstm_prediction(
    actual40_2, lstm_pred40_2,
    actual_label=label_40_2, target_day=40, dataset_label="dataset2", date_label=get_date_label(2, 40),
    output_dir=str(PLOTS_DIR_2),
)

### LSTM Forecast for Day 41 — 10 May 2024 (Dataset 2)


In [ ]:
day40_raw_2 = daily_matrix_2[39]
actual41_2  = daily_matrix_2[40]

day40_in_2 = ((day40_raw_2 - x_mean_2) / x_std_2)[np.newaxis, ..., np.newaxis]   # (1, 1440, 1)

device = next(lstm_model_2.parameters()).device
src    = torch.tensor(day40_in_2, dtype=torch.float32).to(device)

lstm_model_2.eval()
with torch.no_grad():
    pred41_norm_2 = lstm_model_2(src).squeeze().cpu().numpy()   # (1440,)

lstm_pred41_2 = pred41_norm_2 * y_std_2 + y_mean_2

# derive label automatically from the loaded files
label_2 = f"Actual Day 41 ({daily_files_2[-1].stem.replace('_sorted', '')})"  # -> "Actual Day 41 (iisc1310_TECU)"

lstm_rmse_2, lstm_mae_2 = plot_lstm_prediction(
    actual41_2, lstm_pred41_2,
    actual_label=label_2, target_day=41, dataset_label="dataset2", date_label=get_date_label(2, 41),
    output_dir=str(PLOTS_DIR_2),
)

### Transformer Training Workflow for Dataset 2 — Day 40 to Day 41 Forecast (9–10 May 2024)

In [ ]:
transformer_model_2, transformer_history_2 = train_transformer(X_train_2, y_train_2, X_val_2, y_val_2)

### Transformer Loss and Validation Curve — Dataset 2 (9–10 May 2024)

In [ ]:
plot_transformer_loss(transformer_history_2, dataset_label="dataset2", output_dir=str(PLOTS_DIR_2))

### Transformer Forecast for Day 40 — 9th May 2024 (Dataset 2)

In [ ]:
day39_in_t_2 = ((day39_raw_2 - x_mean_2) / x_std_2)[np.newaxis, ..., np.newaxis]   # (1, 1440, 1)

device   = next(transformer_model_2.parameters()).device
src39_t  = torch.tensor(day39_in_t_2, dtype=torch.float32).to(device)

transformer_model_2.eval()
with torch.no_grad():
    trans_pred40_norm_2 = transformer_model_2(src39_t).squeeze().cpu().numpy()   # (1440,)

trans_pred40_2 = trans_pred40_norm_2 * y_std_2 + y_mean_2
trans_rmse_40_2, trans_mae_40_2 = plot_transformer_prediction(
    actual40_2, trans_pred40_2,
    actual_label=label_40_2, target_day=40, dataset_label="dataset2", date_label=get_date_label(2, 40),
    output_dir=str(PLOTS_DIR_2),
)

### Transformer Forecast for Day 41 — 10 May 2024 (Dataset 2)

In [ ]:
day40_in_t_2 = ((day40_raw_2 - x_mean_2) / x_std_2)[np.newaxis, ..., np.newaxis]   # (1, 1440, 1)

device  = next(transformer_model_2.parameters()).device
src_t_2 = torch.tensor(day40_in_t_2, dtype=torch.float32).to(device)

transformer_model_2.eval()
with torch.no_grad():
    trans_pred41_norm_2 = transformer_model_2(src_t_2).squeeze().cpu().numpy()   # (1440,)

trans_pred41_2 = trans_pred41_norm_2 * y_std_2 + y_mean_2
trans_rmse_2, trans_mae_2 = plot_transformer_prediction(
    actual41_2, trans_pred41_2,
    actual_label=label_2, target_day=41, dataset_label="dataset2", date_label=get_date_label(2, 41),
    output_dir=str(PLOTS_DIR_2),
)

### Model Comparison Summary for Day 40/41 Forecast — 9–10 May 2024 (Dataset 2)

In [ ]:
print("=" * 60)
print(f"{'Model':<15} {'Target Day':<12} {'RMSE (TECU)':>12} {'MAE (TECU)':>12}")
print("-" * 60)
print(f"{'LSTM':<15} {'Day 40':<12} {lstm_rmse_40_2:>12.4f} {lstm_mae_40_2:>12.4f}")
print(f"{'Transformer':<15} {'Day 40':<12} {trans_rmse_40_2:>12.4f} {trans_mae_40_2:>12.4f}")
print("-" * 60)
print(f"{'LSTM':<15} {'Day 41':<12} {lstm_rmse_2:>12.4f} {lstm_mae_2:>12.4f}")
print(f"{'Transformer':<15} {'Day 41':<12} {trans_rmse_2:>12.4f} {trans_mae_2:>12.4f}")
print("=" * 60)


### Ionospheric Delay Analysis for LSTM Forecast — Day 40 (9 May 2024, Dataset 2)

In [ ]:
plot_lstm_delay_l1_l5_combined(actual40_2, lstm_pred40_2, target_day=40, dataset_label="dataset2", date_label=get_date_label(2, 40), output_dir=str(PLOTS_DIR_2))

### Ionospheric Delay Analysis for Transformer Forecast — Day 40 (9 May 2024, Dataset 2)

In [ ]:
plot_transformer_delay_l1_l5_combined(actual40_2, trans_pred40_2, target_day=40, dataset_label="dataset2", date_label=get_date_label(2, 40), output_dir=str(PLOTS_DIR_2))

### Ionospheric Delay Analysis for LSTM Forecast — Day 41 (10 May 2024, Dataset 2)

In [ ]:
plot_lstm_delay_l1_l5_combined(actual41_2, lstm_pred41_2, target_day=41, dataset_label="dataset2", date_label=get_date_label(2, 41), output_dir=str(PLOTS_DIR_2))

### Ionospheric Delay Analysis for Transformer Forecast — Day 41 (10 May 2024, Dataset 2)

In [ ]:
plot_transformer_delay_l1_l5_combined(actual41_2, trans_pred41_2, target_day=41, dataset_label="dataset2", date_label=get_date_label(2, 41), output_dir=str(PLOTS_DIR_2))

### Delay Error Metrics for Day 40/41 Forecast — 9–10 May 2024 (Dataset 2)

In [ ]:
iono_actual41_2 = tec_to_iono_delay(actual41_2)
iono_lstm41_2   = tec_to_iono_delay(lstm_pred41_2)
iono_trans41_2  = tec_to_iono_delay(trans_pred41_2)

iono_actual40_2 = tec_to_iono_delay(actual40_2)
iono_lstm40_2   = tec_to_iono_delay(lstm_pred40_2)
iono_trans40_2  = tec_to_iono_delay(trans_pred40_2)

print("Ionospheric Delay Error (L1, vertical) — Day 40 — Dataset 2")
print("=" * 62)
delay_metrics("LSTM",        iono_lstm40_2,  iono_actual40_2)
delay_metrics("Transformer", iono_trans40_2, iono_actual40_2)

print()
print("Ionospheric Delay Error (L1, vertical) — Day 41 — Dataset 2")
print("=" * 62)
delay_metrics("LSTM",        iono_lstm41_2,  iono_actual41_2)
delay_metrics("Transformer", iono_trans41_2, iono_actual41_2)


### Save Forecast Output CSV — Day 40 (9 May 2024, Dataset 2)

In [ ]:
save_delay_csv(actual40_2, lstm_pred40_2, trans_pred40_2, output_dir=OUTPUT_DIR_2, filename="day40_ionospheric_delay.csv")

### Save Forecast Output CSV — Day 41 (10 May 2024, Dataset 2)


In [ ]:
save_delay_csv(actual41_2, lstm_pred41_2, trans_pred41_2, output_dir=OUTPUT_DIR_2, filename="day41_ionospheric_delay.csv")

In [ ]:
lstm_rmse_metric_2 = rmse(lstm_pred41_2, actual41_2)
transformer_rmse_metric_2 = rmse(trans_pred41_2, actual41_2)

print(f"LSTM RMSE:        {lstm_rmse_metric_2:.4f} TECU")
print(f"Transformer RMSE: {transformer_rmse_metric_2:.4f} TECU")


In [ ]:
lstm_correlation_2 = pearson_correlation(lstm_pred41_2, actual41_2)
transformer_correlation_2 = pearson_correlation(trans_pred41_2, actual41_2)

print("Dataset 2 — Day 41")
print(f"LSTM correlation:        {lstm_correlation_2:.4f}")
print(f"Transformer correlation: {transformer_correlation_2:.4f}")


In [ ]:
storm_threshold_2 = float(
    np.percentile(daily_matrix_2[1:30], EVENT_THRESHOLD_PERCENTILE)
)

print(f"Dataset 2 threshold: {storm_threshold_2:.2f} TECU")


In [ ]:
lstm_pod_2 = probability_of_detection(
    lstm_pred41_2, actual41_2, threshold=storm_threshold_2
)
transformer_pod_2 = probability_of_detection(
    trans_pred41_2, actual41_2, threshold=storm_threshold_2
)

print("Dataset 2 — Day 41 POD")
print(f"LSTM POD:        {lstm_pod_2:.4f}")
print(f"Transformer POD: {transformer_pod_2:.4f}")


In [ ]:
lstm_csi_2 = critical_success_index(
    lstm_pred41_2, actual41_2, threshold=storm_threshold_2
)
transformer_csi_2 = critical_success_index(
    trans_pred41_2, actual41_2, threshold=storm_threshold_2
)

print("Dataset 2 — Day 41 CSI")
print(f"LSTM CSI:        {lstm_csi_2:.4f}")
print(f"Transformer CSI: {transformer_csi_2:.4f}")


In [ ]:
lstm_f1_2 = f1_score(
    lstm_pred41_2, actual41_2, threshold=storm_threshold_2
)
transformer_f1_2 = f1_score(
    trans_pred41_2, actual41_2, threshold=storm_threshold_2
)

print("Dataset 2 — Day 41 F1 score")
print(f"LSTM F1:        {lstm_f1_2:.4f}")
print(f"Transformer F1: {transformer_f1_2:.4f}")


In [ ]:
lstm_far_2 = false_alarm_ratio(
    lstm_pred41_2, actual41_2, threshold=storm_threshold_2
)
transformer_far_2 = false_alarm_ratio(
    trans_pred41_2, actual41_2, threshold=storm_threshold_2
)

print("Dataset 2 — Day 41 FAR")
print(f"LSTM FAR:        {lstm_far_2:.4f}")
print(f"Transformer FAR: {transformer_far_2:.4f}")


In [ ]:
# --- RMSE & Correlation on Ionospheric Delay — L1 & L5, LSTM vs Transformer (Dataset 2) ---

from src.utils.iono_delay import compute_all_delays

freqs = {"L1": F_L1, "L5": F_L5}

actual_tec, lstm_tec, trans_tec = actual41_2, lstm_pred41_2, trans_pred41_2

delay_rows = []

for freq_label, freq_hz in freqs.items():

    # Convert TEC -> vertical ionospheric delay (metres) at this frequency
    actual_delay, lstm_delay, trans_delay = compute_all_delays(
        actual_tec, lstm_tec, trans_tec, frequency=freq_hz
    )

    for model_name, pred_delay in (
        ("LSTM", lstm_delay),
        ("Transformer", trans_delay),
    ):
        delay_rmse = rmse(pred_delay, actual_delay)
        delay_corr = pearson_correlation(pred_delay, actual_delay)

        delay_rows.append({
            "Frequency": freq_label,
            "Model": model_name,
            "RMSE (m)": delay_rmse,
            "Correlation": delay_corr,
        })

        print(
            f"{freq_label:<3} | {model_name:<12} "
            f"| RMSE: {delay_rmse:.4f} m | Corr: {delay_corr:.4f}"
        )

iono_delay_metrics_table_2 = pd.DataFrame(delay_rows).round(4)
iono_delay_metrics_table_2

---
## Summary — Metrics Table (Dataset 2, Day 41)
```text
Consolidates every metric computed above (RMSE, Correlation, POD, CSI,
F1 score, FAR) for both models (LSTM, Transformer) for Dataset 2
into a single comparison table, and writes it to CSV for the paper.
```

In [ ]:
metrics_summary = pd.DataFrame({
    "Dataset": ["Dataset 2", "Dataset 2"],
    "Model": ["LSTM", "Transformer"],
    "RMSE (TECU)": [
        lstm_rmse_metric_2, transformer_rmse_metric_2,
    ],
    "Correlation": [
        lstm_correlation_2, transformer_correlation_2,
    ],
    "POD": [
        lstm_pod_2, transformer_pod_2,
    ],
    "CSI": [
        lstm_csi_2, transformer_csi_2,
    ],
    "F1 Score": [
        lstm_f1_2, transformer_f1_2,
    ],
    "FAR": [
        lstm_far_2, transformer_far_2,
    ],
}).round(4)

print("=" * 80)
print("Day 41 Forecast — Summary Metrics Table (Dataset 2)")
print("=" * 80)
print(metrics_summary.to_string(index=False))

summary_csv_path = GENERATED_PLOTS_DIR / "metrics_summary_day41_dataset2.csv"
GENERATED_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
metrics_summary.to_csv(summary_csv_path, index=False)
print(f"\nSaved summary table to: {summary_csv_path}")

metrics_summary
